# Adaptive Aptitude Test Engine - Student Level Model

Trains a **Random Forest** classifier on the self-generated `student_performance.csv` (9,600 rows) to predict a learner's level: **Beginner / Intermediate / Advanced**.

Target accuracy band: **80 - 85 %** (realistic, not overfit).

## 1. Imports

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='darkgrid')
pd.set_option('display.max_columns', 30)

## 2. Load the dataset
Generated by `scripts/generate_student_performance.py` - not downloaded from Kaggle.

In [ ]:
df = pd.read_csv('../data/student_performance.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Data quality checks

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## 4. Exploratory analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='level', data=df, order=['Beginner','Intermediate','Advanced'], palette='viridis')
plt.title('Level distribution')
plt.show()

df['level'].value_counts(normalize=True).round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, ['quant_accuracy','logical_accuracy','verbal_accuracy','di_accuracy']):
    sns.boxplot(x='level', y=col, data=df, order=['Beginner','Intermediate','Advanced'], ax=ax, palette='mako')
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11,8))
sns.heatmap(df.drop(columns=['student_id','level']).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature correlation')
plt.show()

## 5. Features and target

In [ ]:
FEATURES = [
    'tests_taken', 'quant_accuracy', 'logical_accuracy', 'verbal_accuracy', 'di_accuracy',
    'avg_time_per_question', 'attempted_ratio', 'easy_accuracy', 'medium_accuracy',
    'hard_accuracy', 'retake_count', 'consistency_score', 'overall_accuracy',
]
TARGET = 'level'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 6. Train the Random Forest
Depth and leaf size are constrained so the forest generalises instead of memorising label noise.

In [ ]:
model = RandomForestClassifier(
    n_estimators=180,
    max_depth=7,
    min_samples_split=12,
    min_samples_leaf=6,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

## 7. Accuracy

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc*100:.2f}%')
print('Train accuracy: {:.2f}%'.format(accuracy_score(y_train, model.predict(X_train))*100))

assert 0.80 <= acc <= 0.85, 'Accuracy outside the 80-85% target band'

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
print('5-fold CV accuracy:', (scores*100).round(2), ' mean = %.2f%%' % (scores.mean()*100))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
labels = ['Beginner','Intermediate','Advanced']
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion matrix')
plt.show()

## 8. Feature importance

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
plt.figure(figsize=(8,6))
imp.plot(kind='barh', color='#6366f1')
plt.title('Random Forest feature importance')
plt.show()

## 9. Sample prediction

In [ ]:
sample = pd.DataFrame([{
    'tests_taken': 3, 'quant_accuracy': 71.4, 'logical_accuracy': 66.7, 'verbal_accuracy': 75.0,
    'di_accuracy': 33.3, 'avg_time_per_question': 48, 'attempted_ratio': 100, 'easy_accuracy': 85,
    'medium_accuracy': 62, 'hard_accuracy': 40, 'retake_count': 1, 'consistency_score': 58.3,
    'overall_accuracy': 65.0,
}])[FEATURES]
print('Predicted level:', model.predict(sample)[0])
print(dict(zip(model.classes_, model.predict_proba(sample)[0].round(3))))

## 10. Save the model for the Flask app

In [ ]:
bundle = {'model': model, 'features': FEATURES, 'classes': list(model.classes_), 'accuracy': round(float(acc), 4)}
joblib.dump(bundle, '../ml/rf_level_model.pkl')

with open('../ml/metrics.json', 'w') as f:
    json.dump({'accuracy': round(acc*100, 2), 'n_rows': int(df.shape[0]), 'features': FEATURES,
               'classes': list(model.classes_),
               'report': classification_report(y_test, y_pred, output_dict=True)}, f, indent=2)
print('Saved ../ml/rf_level_model.pkl')